In [ ]:
# Performance config
import os

CPU_THREADS = min(32, os.cpu_count() or 32)
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(CPU_THREADS)
os.environ["TOKENIZERS_PARALLELISM"] = "true"

REQUIRE_CUDA = True  # set False for CPU-only notebooks

print(f"CPU threads set to: {CPU_THREADS}")

try:
    import torch
except Exception as e:
    torch = None
    if REQUIRE_CUDA:
        raise RuntimeError("CUDA required but torch is not available.") from e

if torch is not None:
    torch.set_num_threads(CPU_THREADS)
    torch.set_num_interop_threads(min(4, CPU_THREADS))
    if REQUIRE_CUDA and not torch.cuda.is_available():
        raise RuntimeError("CUDA required but not available.")
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        print("CUDA device:", torch.cuda.get_device_name(0))
    else:
        print("CUDA not available; running on CPU.")


# Kvasir-VQA x1 — Image-only baseline (frozen Swin + Logistic Regression)

Extract frozen Swin Transformer embeddings for each image, cache them, and train a multinomial logistic regression classifier on top-K answers. Mirrors the ViT frozen + LogReg baseline.


In [1]:
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoImageProcessor, SwinModel

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.preprocessing import StandardScaler

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


2026-01-27 03:37:58.586230: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-01-27 03:37:58.586266: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-01-27 03:37:58.587240: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-27 03:37:58.593409: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-27 03:37:59.539407: W tensorflow/compiler/tf2

In [2]:
# Paths & config

def find_dataset_root() -> Path:
    start = Path.cwd().resolve()
    for p in [start] + list(start.parents):
        if (p / "0_dataset_prep").exists():
            return p
    raise RuntimeError(f"Could not locate dataset root containing '0_dataset_prep'. cwd={start}")

DATA_ROOT = find_dataset_root()
META_CSV = DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "metadata_enriched.csv"
OUT_DIR = DATA_ROOT / "2_modeling" / "02_image_only" / "out"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "microsoft/swin-tiny-patch4-window7-224"
BATCH_SIZE = 16
NUM_WORKERS = 0
TOP_K = 200

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TORCH_DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

print("Data root:", DATA_ROOT)
print("Metadata:", META_CSV)
print("Out dir:", OUT_DIR)
print("Device:", DEVICE)
print("Torch dtype:", TORCH_DTYPE)


Data root: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1
Metadata: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/0_dataset_prep/out/metadata/metadata_enriched.csv
Out dir: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/2_modeling/02_image_only/out
Device: cuda
Torch dtype: torch.float16


In [3]:
# Load metadata and splits
meta = pd.read_csv(META_CSV)

# Resolve image paths relative to dataset root if needed
images_base = DATA_ROOT / "0_dataset_prep"
meta["image_path"] = meta["image_path"].apply(
    lambda p: str((images_base / p).resolve()) if not Path(p).is_absolute() else p
)

meta["answer_norm"] = meta["answer"].fillna("").astype(str).str.lower().str.strip()

if "split" not in meta.columns:
    raise RuntimeError("Missing 'split' column. Run dataset prep split step first.")

train_df = meta[meta["split"] == "train"].reset_index(drop=True)
val_df = meta[meta["split"] == "validation"].reset_index(drop=True)
test_df = meta[meta["split"] == "test"].reset_index(drop=True)

print({"train": len(train_df), "val": len(val_df), "test": len(test_df)})


{'train': 46966, 'val': 5931, 'test': 5952}


In [4]:
# Top-K answers from train split
answer_counts = train_df["answer_norm"].value_counts()
TOP_K_ANSWERS = answer_counts.head(TOP_K).index.tolist()

train_k = train_df[train_df["answer_norm"].isin(TOP_K_ANSWERS)].reset_index(drop=True)
val_k = val_df[val_df["answer_norm"].isin(TOP_K_ANSWERS)].reset_index(drop=True)
test_k = test_df[test_df["answer_norm"].isin(TOP_K_ANSWERS)].reset_index(drop=True)

print("Top-K answers:", len(TOP_K_ANSWERS))
print({"train": len(train_k), "val": len(val_k), "test": len(test_k)})


Top-K answers: 200
{'train': 46598, 'val': 5884, 'test': 5893}


In [5]:
# Utilities

def compute_metrics(y_true, y_pred):
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro"))
    }
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    return metrics, report


def save_split(prefix, split_name, df, y_pred, metrics, report):
    pred_df = df[["answer_norm"]].copy()
    pred_df["pred"] = y_pred
    pred_df.to_csv(OUT_DIR / f"{prefix}_pred_{split_name}.csv", index=False)
    with open(OUT_DIR / f"{prefix}_metrics_{split_name}.json", "w") as f:
        json.dump({"metrics": metrics, "report": report}, f, indent=2)


In [6]:
# Image embedding extraction using Swin
class ImageDS(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = row["img_id"]
        img = Image.open(row["image_path"]).convert("RGB")
        return img_id, img


def collate_fn(batch):
    ids = [b[0] for b in batch]
    images = [b[1] for b in batch]
    inputs = processor(images=images, return_tensors="pt")
    return ids, inputs["pixel_values"]


def compute_embeddings(unique_df):
    ds = ImageDS(unique_df)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_fn)
    emb_map = {}
    with torch.no_grad():
        for ids, pixels in tqdm(dl, desc="Swin embed"):
            pixels = pixels.to(DEVICE)
            out = swin(pixels)
            # SwinModel returns pooler_output (global avg pool) when available
            if hasattr(out, "pooler_output") and out.pooler_output is not None:
                emb_batch = out.pooler_output
            else:
                emb_batch = out.last_hidden_state.mean(dim=1)
            for i, img_id in enumerate(ids):
                emb_map[img_id] = emb_batch[i].cpu().numpy()
    return emb_map


processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
try:
    swin = SwinModel.from_pretrained(MODEL_NAME, torch_dtype=TORCH_DTYPE).to(DEVICE)
except RuntimeError as e:
    print(f"Falling back to CPU for Swin (error: {e}).")
    DEVICE = torch.device("cpu")
    TORCH_DTYPE = torch.float32
    swin = SwinModel.from_pretrained(MODEL_NAME, torch_dtype=TORCH_DTYPE).to(DEVICE)

swin.eval()

EMB_PATH = OUT_DIR / "swin_embeddings.npz"
unique_imgs = pd.concat([train_k, val_k, test_k])[ ["img_id", "image_path"] ].drop_duplicates()

if EMB_PATH.exists():
    data = np.load(EMB_PATH, allow_pickle=True)
    img_ids = data["img_ids"].tolist()
    embeddings = data["embeddings"]
    emb_map = {img_id: embeddings[i] for i, img_id in enumerate(img_ids)}
    print("Loaded cached embeddings:", len(emb_map))
else:
    emb_map = compute_embeddings(unique_imgs)
    img_ids = list(emb_map.keys())
    embeddings = np.stack([emb_map[i] for i in img_ids])
    np.savez(EMB_PATH, img_ids=np.array(img_ids), embeddings=embeddings)
    print("Saved embeddings:", EMB_PATH)


def build_X(df):
    return np.stack([emb_map[i] for i in df["img_id"].tolist()])


preprocessor_config.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/113M [00:00<?, ?B/s]

Falling back to CPU for Swin (error: CUDA error: out of memory
Search for `cudaErrorMemoryAllocation' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
).


Swin embed:   0%|          | 0/407 [00:00<?, ?it/s]

Saved embeddings: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/2_modeling/02_image_only/out/swin_embeddings.npz


In [7]:
# Build feature matrices
X_train = build_X(train_k)
X_val = build_X(val_k) if len(val_k) else None
X_test = build_X(test_k) if len(test_k) else None

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
if X_val is not None:
    X_val = scaler.transform(X_val)
if X_test is not None:
    X_test = scaler.transform(X_test)

y_train = train_k["answer_norm"].values
y_val = val_k["answer_norm"].values if len(val_k) else None
y_test = test_k["answer_norm"].values if len(test_k) else None

print("Feature dim:", X_train.shape[1])


Feature dim: 768


In [8]:
# Train logistic regression classifier
clf = LogisticRegression(max_iter=1000, n_jobs=-1)
clf.fit(X_train, y_train)


def eval_split(X, y_true, split_name):
    y_pred = clf.predict(X)
    metrics, report = compute_metrics(y_true, y_pred)
    save_split("swin", split_name, pd.DataFrame({"answer_norm": y_true}), y_pred, metrics, report)
    print(split_name, metrics)
    return metrics

train_metrics = eval_split(X_train, y_train, "train")
if X_val is not None:
    val_metrics = eval_split(X_val, y_val, "val")
else:
    val_metrics = None
if X_test is not None:
    test_metrics = eval_split(X_test, y_test, "test")
else:
    test_metrics = None


/home/aristotle/anaconda3/envs/vqa-rag/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:470: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


train {'accuracy': 0.2826086956521739, 'macro_f1': 0.015233737869619249}
val {'accuracy': 0.21872875594833446, 'macro_f1': 0.018635326473345322}
test {'accuracy': 0.22229764126930257, 'macro_f1': 0.014465193382405862}


In [9]:
# Save model artifacts
from joblib import dump

model_path = OUT_DIR / "swin_logreg.joblib"
dump({
    "clf": clf,
    "scaler": scaler,
    "emb_path": EMB_PATH,
    "model_name": MODEL_NAME,
}, model_path)
print("Saved model to", model_path)


Saved model to /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/2_modeling/02_image_only/out/swin_logreg.joblib


In [10]:
# Summary
import pandas as pd
summary_rows = []
for name, mets in [("train", train_metrics), ("val", val_metrics), ("test", test_metrics)]:
    if mets:
        summary_rows.append({"split": name, **mets})

if summary_rows:
    display(pd.DataFrame(summary_rows).set_index("split").round(4))
else:
    print("No metrics collected.")


,accuracy,macro_f1
split,,
train,0.2826,0.0152
val,0.2187,0.0186
test,0.2223,0.0145
